# 03 — Feature Importance, Feature Reduction and Class-Imbalance Analysis

Clean rerun using the 40,000-row source dataset and the freshly selected **pre-call** hyperparameters from Notebook 02.

This notebook is self-contained for execution:
- no `data/`, `results/`, or `figures/` folder dependency;
- no cached CSV/JSON/NPY result dependency;
- no generated-file writes;
- `duration` is excluded from every deployable analysis;
- all figures and tables are displayed in the notebook only.

## 1. Setup, canonical split and upstream pre-call parameters

The same deterministic 80/20 stratified split (`SEED=42`) used in Notebook 02 is recreated here.  
Notebook 02 selected the pre-call settings by validation PR-AUC with ROC-AUC as tie-breaker.  
The selected settings are transferred here **without retuning**.

**Coding step.** Load the data, recreate the canonical split and transfer the freshly selected pre-call LR/HGB settings from Notebook 02 without retuning.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
import warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

warnings.filterwarnings("ignore")
SEED = 42
DATA_SOURCE = "term-deposit-marketing-2020-labelled.csv"

NUM = ["age","balance","day","campaign"]
CAT = ["job","marital","education","default","housing","loan","contact","month"]
PRE = NUM + CAT

SELECTED = {
    "LR": {"C": 8.483428982440726, "penalty": "l2", "class_weight": None},
    "HGB": {"max_iter": 200, "max_depth": 8, "learning_rate": 0.05, "max_leaf_nodes": 15, "l2_regularization": 0.0, "min_samples_leaf": 20, "class_weight": None, "early_stopping": True}
}

df = pd.read_csv(DATA_SOURCE)
if "y_binary" in df.columns: df = df.drop(columns="y_binary")
df["y_binary"] = df["y"].eq("yes").astype(int)
y = df["y_binary"].to_numpy(); idx = np.arange(len(df))
dev_idx, test_idx = train_test_split(idx, test_size=0.20, stratify=y, random_state=SEED)
print("Dataset:", df.drop(columns="y_binary").shape)
print("Development:", len(dev_idx), "Final holdout:", len(test_idx))
print("Duration present in deployable features:", "duration" in PRE)
print(json.dumps(SELECTED, indent=2))

## 2. Reusable model and metric functions

Categorical variables are one-hot encoded and numeric variables are standardised. HGB receives a dense transformed matrix; Logistic Regression uses the same deployable predictors.

**Coding step.** Define common preprocessing, model constructors, six headline metrics and association helpers.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
def preprocessor(features):
    nums=[c for c in features if c in NUM]; cats=[c for c in features if c in CAT]
    return ColumnTransformer([("num",StandardScaler(),nums),("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),cats)],remainder="drop")
def make_lr(params=None):
    p=dict(SELECTED["LR"] if params is None else params); return LogisticRegression(solver="liblinear",max_iter=2000,random_state=SEED,**p)
def make_hgb(params=None):
    p=dict(SELECTED["HGB"] if params is None else params); return HistGradientBoostingClassifier(random_state=SEED,**p)
def make_pipe(kind,features,params=None):
    return Pipeline([("prep",preprocessor(features)),("model",make_lr(params) if kind=="LR" else make_hgb(params))])
def metrics(y_true,prob,threshold=0.5):
    pred=(np.asarray(prob)>=threshold).astype(int); return {"accuracy":accuracy_score(y_true,pred),"precision":precision_score(y_true,pred,zero_division=0),"recall":recall_score(y_true,pred,zero_division=0),"f1":f1_score(y_true,pred,zero_division=0),"pr_auc":average_precision_score(y_true,prob),"roc_auc":roc_auc_score(y_true,prob)}
def cramer_v(x,target):
    t=pd.crosstab(x,target); n=t.to_numpy().sum(); chi=chi2_contingency(t,correction=False)[0]; r,k=t.shape; phi=chi/n; pc=max(0,phi-((k-1)*(r-1))/(n-1)); rc=r-((r-1)**2)/(n-1); kc=k-((k-1)**2)/(n-1); den=min(rc-1,kc-1); return np.sqrt(pc/den) if den>0 else 0.0

## 3. Feature importance

Direct target association, grouped LR coefficient magnitude and HGB permutation importance answer different questions. The feature-reduction order is defined **only by HGB held-out permutation importance**, not by a blended rank.

**Coding step.** Compute direct association, grouped LR coefficient magnitude and held-out HGB permutation importance as distinct evidence types.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
assoc={f:abs(df[f].corr(df["y_binary"])) for f in NUM}
assoc.update({f:cramer_v(df[f],df["y"]) for f in CAT})
lr_full=make_pipe("LR",PRE); lr_full.fit(df.loc[dev_idx,PRE],y[dev_idx])
prep=lr_full.named_steps["prep"]; coef=np.abs(lr_full.named_steps["model"].coef_[0]); names=prep.get_feature_names_out(); lr_group={}
for f in PRE:
    mask=np.array([n==f"num__{f}" for n in names]) if f in NUM else np.array([n.startswith(f"cat__{f}_") for n in names]); lr_group[f]=coef[mask].max() if mask.any() else np.nan
hgb_full=make_pipe("HGB",PRE); hgb_full.fit(df.loc[dev_idx,PRE],y[dev_idx])
perm=permutation_importance(hgb_full,df.loc[test_idx,PRE],y[test_idx],scoring="average_precision",n_repeats=8,random_state=SEED,n_jobs=1)
importance=pd.DataFrame({"feature":PRE,"association_strength":[assoc[f] for f in PRE],"lr_importance":[lr_group[f] for f in PRE],"hgb_permutation":perm.importances_mean})
importance["association_rank"]=importance.association_strength.rank(method="min",ascending=False).astype(int); importance["lr_rank"]=importance.lr_importance.rank(method="min",ascending=False).astype(int); importance["hgb_rank"]=importance.hgb_permutation.rank(method="min",ascending=False).astype(int)
importance=importance.sort_values("hgb_rank").reset_index(drop=True); display(importance.round(6))

### 3.1 Notebook-generated feature-importance bar plots

**Coding step.** Plot grouped LR coefficient magnitude and held-out HGB permutation importance as two separate horizontal bar charts using the `importance` table calculated above.

**Why this step matters.** The measures have different meanings and scales, so they should not be blended into a synthetic score. These notebook outputs are the canonical source for the corresponding Technical Report figures. `duration` cannot appear because only the 12 valid pre-call predictors are fitted.

In [ ]:
lr_plot = importance[["feature","lr_importance"]].sort_values("lr_importance")
fig, ax = plt.subplots(figsize=(8.5,5.8))
ax.barh(lr_plot["feature"], lr_plot["lr_importance"])
ax.set(title="Logistic Regression Feature Importance", xlabel="Maximum absolute encoded coefficient", ylabel="")
ax.margins(x=0.05)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

hgb_plot = importance[["feature","hgb_permutation"]].sort_values("hgb_permutation")
fig, ax = plt.subplots(figsize=(8.5,5.8))
ax.barh(hgb_plot["feature"], hgb_plot["hgb_permutation"])
ax.axvline(0, linewidth=1)
ax.set(title="HistGradientBoosting Permutation Importance", xlabel="Mean decrease in holdout PR-AUC after permutation", ylabel="")
ax.margins(x=0.05)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Feature-reduction experiment

Top-3, Top-4 and Top-5 subsets are nested according to the fresh HGB permutation ranking. Each subset is evaluated on the untouched 20% holdout with the same tuned structural settings; there is no subset-specific hyperparameter search.

**Coding step.** Visualise all six headline metrics for the feature-reduction comparison from the notebook result table.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
ranked=importance.sort_values("hgb_rank")["feature"].tolist(); feature_sets={"Top 3":ranked[:3],"Top 4":ranked[:4],"Top 5":ranked[:5],"All 12":PRE}
for k,v in feature_sets.items(): print(k,":",v)
rows=[]
for label,features in feature_sets.items():
    for kind in ["LR","HGB"]:
        est=make_pipe(kind,features); est.fit(df.loc[dev_idx,features],y[dev_idx]); p=est.predict_proba(df.loc[test_idx,features])[:,1]; rows.append({"feature_set":label,"model":kind,**metrics(y[test_idx],p)})
feature_reduction=pd.DataFrame(rows); display(feature_reduction.round(6))

### 4.1 Six-metric feature-reduction panel

Because the feature-reduction table contains all six headline metrics, it is paired with a 2×3 panel. The experiment is diagnostic and does not redefine the final feature set from one metric alone.

**Coding step.** Visualise all six headline metrics for the feature-reduction comparison from the notebook result table.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
metric_order=["accuracy","precision","recall","f1","pr_auc","roc_auc"]
fig,axes=plt.subplots(2,3,figsize=(15,8)); axes=axes.ravel()
for ax,m in zip(axes,metric_order):
    pivot=feature_reduction.pivot(index="feature_set",columns="model",values=m).reindex(["Top 3","Top 4","Top 5","All 12"]); pivot.plot(kind="bar",ax=ax,legend=False); ax.set_title(m.replace("_"," ").upper()); ax.set_xlabel(""); ax.tick_params(axis="x",rotation=30); ax.grid(axis="y",alpha=.25)
handles,labels=axes[0].get_legend_handles_labels(); fig.suptitle("Feature Reduction - Six-Metric Holdout Comparison",y=0.995); fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,0.955),ncol=2,frameon=False); fig.tight_layout(rect=[0,0,1,0.91]); plt.show()


## 5. Six class-imbalance strategies

All strategies use the same full 12-feature pre-call representation and Notebook-02 structural hyperparameters. Threshold optimisation changes hard classifications but does not change PR-AUC or ROC-AUC.

**Coding step.** Compare no adjustment, weighting, over/undersampling, SMOTENC and nested threshold optimisation using the same pre-call model structures.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
from imblearn.over_sampling import RandomOverSampler,SMOTENC
from imblearn.under_sampling import RandomUnderSampler
Xdev=df.loc[dev_idx,PRE].copy(); ydev=y[dev_idx]; Xtest=df.loc[test_idx,PRE].copy(); ytest=y[test_idx]
def fit_predict(kind,params,Xtr,ytr):
    est=make_pipe(kind,PRE,params); est.fit(Xtr,ytr); return est.predict_proba(Xtest)[:,1]
def balanced(kind): p=dict(SELECTED[kind]); p["class_weight"]="balanced"; return p
def smote(kind):
    Xc=Xdev.copy(); [Xc.__setitem__(c,Xc[c].astype("category")) for c in CAT]; Xr,yr=SMOTENC(categorical_features=CAT,random_state=SEED).fit_resample(Xc,ydev); [Xr.__setitem__(c,Xr[c].astype(str)) for c in CAT]; return fit_predict(kind,SELECTED[kind],Xr,yr)
def threshold_oof(kind):
    skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED); oof=np.zeros(len(Xdev))
    for tr,va in skf.split(Xdev,ydev):
        est=make_pipe(kind,PRE,SELECTED[kind]); est.fit(Xdev.iloc[tr],ydev[tr]); oof[va]=est.predict_proba(Xdev.iloc[va])[:,1]
    cand=np.linspace(.005,.995,991); f1s=np.array([f1_score(ydev,oof>=t,zero_division=0) for t in cand]); t=float(cand[np.flatnonzero(f1s==f1s.max())[0]]); est=make_pipe(kind,PRE,SELECTED[kind]); est.fit(Xdev,ydev); return est.predict_proba(Xtest)[:,1],t,float(f1s.max())
imbalance_rows=[]; threshold_audit=[]
for kind in ["LR","HGB"]:
    p=fit_predict(kind,SELECTED[kind],Xdev,ydev); imbalance_rows.append({"model":kind,"strategy":"none","threshold":.5,**metrics(ytest,p)})
    p=fit_predict(kind,balanced(kind),Xdev,ydev); imbalance_rows.append({"model":kind,"strategy":"class_weight","threshold":.5,**metrics(ytest,p)})
    for name,sampler in [("oversampling",RandomOverSampler(random_state=SEED)),("undersampling",RandomUnderSampler(random_state=SEED))]:
        Xr,yr=sampler.fit_resample(Xdev,ydev); p=fit_predict(kind,SELECTED[kind],Xr,yr); imbalance_rows.append({"model":kind,"strategy":name,"threshold":.5,**metrics(ytest,p)})
    p=smote(kind); imbalance_rows.append({"model":kind,"strategy":"SMOTENC","threshold":.5,**metrics(ytest,p)})
    p,t,oof_f1=threshold_oof(kind); imbalance_rows.append({"model":kind,"strategy":"threshold_tuning","threshold":t,**metrics(ytest,p,t)}); threshold_audit.append({"model":kind,"selected_threshold":t,"development_oof_f1":oof_f1})
imbalance=pd.DataFrame(imbalance_rows); threshold_audit=pd.DataFrame(threshold_audit); display(imbalance.round(6)); display(threshold_audit.round(6))

## 6. Required six-metric comparison panel

Every model/strategy comparison table containing Accuracy, Precision, Recall, F1, PR-AUC and ROC-AUC is paired with the same six metrics graphically.

**Coding step.** Visualise all six headline metrics for the feature-reduction comparison from the notebook result table.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(15,8)); axes=axes.ravel()
for ax,m in zip(axes,metric_order):
    imbalance.pivot(index="strategy",columns="model",values=m).plot(kind="bar",ax=ax,legend=False); ax.set_title(m.replace("_"," ").upper()); ax.set_xlabel(""); ax.tick_params(axis="x",rotation=45); ax.grid(axis="y",alpha=.25)
handles,labels=axes[0].get_legend_handles_labels(); fig.suptitle("Class-Imbalance Strategies - Six-Metric Holdout Comparison",y=0.995); fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,0.955),ncol=2,frameon=False); fig.tight_layout(rect=[0,0,1,0.91]); plt.show()


## 7. Notebook-03 conclusion

Feature reduction is a ranking/stability experiment, not evidence that omitted predictors are invalid. Class-imbalance interventions are compared primarily on ranking and secondarily on precision/recall/F1. `duration` is absent throughout.

**Coding step.** Print compact feature and imbalance cross-checks for report traceability.

**Why this step matters.** Keeping the rationale next to the code makes the analytical sequence, validation logic and interpretation auditable.

In [ ]:
print("Top-five HGB permutation features:",ranked[:5])
display(imbalance.loc[imbalance.groupby("model")["pr_auc"].idxmax(),["model","strategy","pr_auc","roc_auc","precision","recall","f1"]].reset_index(drop=True).round(6))
display(imbalance.loc[imbalance.groupby("model")["f1"].idxmax(),["model","strategy","threshold","f1","precision","recall","pr_auc","roc_auc"]].reset_index(drop=True).round(6))
print("Duration used:","duration" in PRE)